# Spatial Batching Visualization

Visualizes how the KD-tree recursive bisection spatial batching assigns
DRB HRU polygons to batches for memory-efficient raster processing.

The pipeline processes the full target fabric (no domain bbox filter),
so batching operates on all 765 HRUs.

In [ ]:
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt

from hydro_param.batching import spatial_batch

In [ ]:
# Load fabric (full extent — no domain filter, matching pipeline default)
gdf = gpd.read_file("../data/pywatershed_gis/drb_2yr/nhru.gpkg")
print(f"Fabric: {len(gdf)} features, CRS={gdf.crs}")

In [ ]:
# Apply spatial batching with batch_size=80
batched = spatial_batch(gdf, batch_size=80)
n_batches = batched["batch_id"].nunique()
print(f"Batches: {n_batches}")
print(batched.groupby("batch_id").size().rename("count"))

In [ ]:
# Plot in WGS84
batched_4326 = batched.to_crs("EPSG:4326")

fig, ax = plt.subplots(1, 1, figsize=(10, 14))
batched_4326.plot(
    column="batch_id",
    categorical=True,
    cmap="Set3",
    edgecolor="gray",
    linewidth=0.3,
    legend=True,
    legend_kwds={"title": "Batch ID", "loc": "lower right"},
    ax=ax,
)

# Add batch labels at centroid of each group
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*geographic CRS.*centroid.*")
    for batch_id in sorted(batched_4326["batch_id"].unique()):
        group = batched_4326[batched_4326["batch_id"] == batch_id]
        cx = group.geometry.centroid.x.mean()
        cy = group.geometry.centroid.y.mean()
        count = len(group)
        ax.annotate(
            f"B{batch_id}\n({count})",
            xy=(cx, cy),
            fontsize=9,
            fontweight="bold",
            ha="center",
            va="center",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
        )

ax.set_title(
    f"DRB Spatial Batching: {len(batched_4326)} HRUs \u2192 {n_batches} batches (batch_size=80)"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# Show bbox extent for each batch
from hydro_param.pipeline import _buffered_bbox

header = (
    f"{'Batch':>5} {'Count':>5} {'West':>8} {'South':>7}"
    f" {'East':>8} {'North':>7} {'dLon':>6} {'dLat':>6}"
)
print(header)
print("-" * 60)
for batch_id in sorted(batched["batch_id"].unique()):
    group = batched[batched["batch_id"] == batch_id]
    bb = _buffered_bbox(group)
    dlon = bb[2] - bb[0]
    dlat = bb[3] - bb[1]
    print(
        f"{batch_id:>5} {len(group):>5}"
        f" {bb[0]:>8.3f} {bb[1]:>7.3f}"
        f" {bb[2]:>8.3f} {bb[3]:>7.3f}"
        f" {dlon:>6.3f} {dlat:>6.3f}"
    )